# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [46]:
# Initialization

load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")
    
MODEL = "llama-3.3-70b-versatile"

Groq API Key exists and begins gsk_TdT2


In [47]:
#Connect to Groq

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [6]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [7]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [10]:
get_ticket_price("Maldives")

Tool called for city Maldives


'The price of a ticket to Maldives is Unknown ticket price'

In [11]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [12]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [13]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [14]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [17]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)

        for message in messages:
            print(message)
            
        response = groq.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [19]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Tool called for city Tokyo
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'Hello'}
{'role': 'assistant', 'content': "I'm happy to assist you with any FlightAI-related inquiries you may have."}
{'role': 'user', 'content': "I'd like a trip to Tokyo"}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='phzeehjxt', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to Tokyo is $1400', 'tool_call_id': 'phzeehjxt'}
Tool called for city Tokyo
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 s

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [20]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [24]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [25]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Paris


In [23]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [26]:
import sqlite3


In [27]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [28]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [29]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [30]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [31]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [32]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [33]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London


## Exercise

Add a tool to set the price of a ticket!

In [48]:
import json

# DATA (Dictionary version)

ticket_prices = {
    "London": 799,
    "Paris": 699,
    "Tokyo": 999,
    "New York": 899
}

In [49]:
#Tools - GET & SET function

def get_ticket_price(destination_city):
    return ticket_prices.get(destination_city, "Unknown destination")

def set_ticket_price(city, price):
    ticket_prices[city] = price
    return f"Price updated for {city} to {price}"

In [52]:
#Tool schema for GET function

price_function = {
    "name": "get_ticket_price",
    "description": "Get the ticket price for a given destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "City name like London, Paris, Tokyo (use key: destination_city exactly)"
            }
        },
        "required": ["destination_city"]
    }
}

In [53]:
#Tool schema for SET function

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the ticket price for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name"
            },
            "price": {
                "type": "integer",
                "description": "New ticket price"
            }
        },
        "required": ["city", "price"]
    }
}

In [54]:
#Tools list

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

In [63]:
#System prompt

system_message = """
You are a helpful airline assistant.

When calling tools:
- Always return valid JSON arguments
- Never return null
- Never leave arguments empty
- Use exact parameter names only
- Do NOT write <function=...>

If unsure, ask the user instead of calling a tool.
"""

In [64]:
#Chat function

def safe_parse_arguments(raw_args):
    try:
        parsed = json.loads(raw_args) if raw_args else {}
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


# function registry (no if-else needed)
function_registry = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price
}


def chat(message, history):
    messages = [{"role": "system", "content": system_message}]

    # add history
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": message})

    # ---------------- FIRST CALL ----------------
    response = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    # ---------------- TOOL HANDLING ----------------
    if msg.tool_calls:

        for tool_call in msg.tool_calls:

            func_name = tool_call.function.name

            # SAFE ARG PARSE (NEVER RETURNS NONE)
            args = safe_parse_arguments(tool_call.function.arguments)

            print("DEBUG tool:", func_name, args)   # optional debug

            # GET FUNCTION FROM REGISTRY
            func = function_registry.get(func_name)

            if not func:
                return f"Error: Unknown function → {func_name}"

            # SAFE FUNCTION CALL
            try:
                result = func(**args)
            except TypeError as e:
                return f"Error: Bad arguments {args} → {str(e)}"
            except Exception as e:
                return f"Error while executing function → {str(e)}"

            # APPEND TOOL CALL MESSAGE (CORRECT FORMAT)
            messages.append({
                "role": "assistant",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": func_name,
                            "arguments": json.dumps(args)
                        }
                    }
                ]
            })

            # APPEND TOOL RESULT
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

        # ---------------- SECOND CALL ----------------
        final_response = groq.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return final_response.choices[0].message.content

    # ---------------- NORMAL RESPONSE ----------------
    return msg.content

In [65]:
#Gradio UI

gr.ChatInterface(
    fn=chat,
    title="✈️ FlightAI Assistant",
    description="Ask about ticket prices or update them!",
    theme="soft"
).launch()

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


DEBUG tool: get_ticket_price {'destination_city': 'London'}
DEBUG tool: set_ticket_price {'city': 'Paris', 'price': 1499}
DEBUG tool: set_ticket_price {'city': 'Paris', 'price': 1499}
DEBUG tool: get_ticket_price {'destination_city': 'Paris'}


<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>